# MSFormer — Results Analysis

Loads results from `results/` and produces the full comparison table, Mann-Whitney tests, and figures.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..') / 'src'))

from msformer.evaluation.stats import compare_conditions, results_table

sns.set_theme(style='whitegrid', font_scale=1.2)
RESULTS_ROOT = Path('../results')

## Load results

In [ ]:
def load_results(task: str, variant: str) -> dict:
    path = RESULTS_ROOT / task / 'transformer_results.json'
    with open(path) as f:
        tr = json.load(f)
    # Extract balanced_accuracy for the specified reduction variant
    return {cond: tr[cond][variant]['balanced_accuracy'] for cond in tr}

def load_baselines(task: str, variant: str) -> dict:
    path = RESULTS_ROOT / task / f'baseline_{variant}_results.json'
    with open(path) as f:
        bl = json.load(f)
    return {k: bl[k]['balanced_accuracy'] for k in bl}

# Olive oil (sum reduction)
oo_tr = load_results('olive_oil', 'sum')
oo_bl = load_baselines('olive_oil', 'sum')
oo_all = {**oo_tr, **oo_bl}

# Hemp seed
hemp_tr = load_results('hemp_seed', 'gcms')
hemp_bl = load_baselines('hemp_seed', 'gcms')
hemp_all = {**hemp_tr, **hemp_bl}

## Summary table (Table 1)

In [ ]:
def make_summary_row(cond, scores):
    arr = np.array(scores)
    return f"{arr.mean():.3f} ± {arr.std():.3f}"

conditions = [
    ('from_scratch', 'A. From-scratch transformer'),
    ('msm_finetune', 'B. Pretrained MSM → fine-tune'),
    ('cont_finetune', 'C. Pretrained Contrastive → fine-tune'),
    ('linear_probe_msm', 'D. Pretrained MSM → linear probe'),
    ('linear_probe_cont', 'D. Pretrained Contrastive → linear probe'),
    ('plsda', 'PLS-DA (baseline)'),
    ('rf', 'Random Forest (baseline)'),
    ('svm', 'SVM-RBF (baseline)'),
]

rows = []
for key, label in conditions:
    row = {'Condition': label}
    if key in oo_all:
        row['Olive Oil (BA)'] = make_summary_row(key, oo_all[key])
    else:
        row['Olive Oil (BA)'] = '—'
    if key in hemp_all:
        row['Hemp Seed (BA)'] = make_summary_row(key, hemp_all[key])
    else:
        row['Hemp Seed (BA)'] = '—'
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Condition')
summary_df

## Mann-Whitney U tests (Bonferroni corrected)

In [ ]:
print('=== Olive Oil ===')
oo_comp = compare_conditions(oo_all)
display(oo_comp[['condition_a','condition_b','mean_a','mean_b','p_corrected','effect_r','significant']])

print('\n=== Hemp Seed ===')
hemp_comp = compare_conditions(hemp_all)
display(hemp_comp[['condition_a','condition_b','mean_a','mean_b','p_corrected','effect_r','significant']])

## Box plots (Figure 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_conditions = ['from_scratch','msm_finetune','cont_finetune','linear_probe_msm','plsda','rf','svm']
short_labels = ['Scratch','MSM-FT','Cont-FT','MSM-Probe','PLS-DA','RF','SVM']
colors = ['#7fc97f','#beaed4','#fdc086','#ffff99','#386cb0','#f0027f','#bf5b17']

for ax, all_res, title in zip(axes, [oo_all, hemp_all], ['Olive Oil (GC-IMS)', 'Hemp Seed (GC/MS)']):
    data = [np.array(all_res[c]) if c in all_res else np.array([np.nan]) for c in plot_conditions]
    bp = ax.boxplot(data, labels=short_labels, patch_artist=True, medianprops=dict(color='black',lw=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_ylabel('Balanced Accuracy')
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, ls='--', color='grey', lw=1, label='Chance')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../results/figure1_balanced_accuracy.pdf', bbox_inches='tight')
plt.show()

## Olive oil: sum vs apex reduction comparison (Figure 2)

In [ ]:
oo_sum = load_results('olive_oil', 'sum')
oo_apex = load_results('olive_oil', 'apex')

comp_conditions = ['from_scratch', 'msm_finetune', 'cont_finetune']
x = np.arange(len(comp_conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
means_sum = [np.mean(oo_sum[c]) if c in oo_sum else 0 for c in comp_conditions]
stds_sum = [np.std(oo_sum[c]) if c in oo_sum else 0 for c in comp_conditions]
means_apex = [np.mean(oo_apex[c]) if c in oo_apex else 0 for c in comp_conditions]
stds_apex = [np.std(oo_apex[c]) if c in oo_apex else 0 for c in comp_conditions]

ax.bar(x - width/2, means_sum, width, yerr=stds_sum, label='Sum reduction', capsize=4, color='#4393c3', alpha=0.85)
ax.bar(x + width/2, means_apex, width, yerr=stds_apex, label='Apex reduction', capsize=4, color='#d6604d', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(['From scratch', 'MSM fine-tune', 'Contrastive fine-tune'])
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Olive Oil: IMS reduction strategy comparison')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('../results/figure2_ims_reduction.pdf', bbox_inches='tight')
plt.show()

## Pre-registered predictions vs outcomes

In [ ]:
predictions = {
    'If B or C beats A on BOTH tasks': 'Pretraining transfers across reference→whole-sample gap',
    'If B or C beats A on ONE task only': 'Partial transfer — structure-dependent',
    'If A matches B and C': 'Negative result: reference spectra do not transfer (publishable)',
    'If D underperforms B/C': 'Representations are useful but require adaptation (partial transfer)',
}
print('Pre-registered outcome predictions (avoid post-hoc rationalisation):')
for pred, interp in predictions.items():
    print(f'  [{pred}]')
    print(f'    → {interp}\n')

# Fill in once results are available:
print('--- Actual outcomes (fill in after running experiments) ---')
print('TODO: compare conditions and select the matching prediction above.')